**Goal.** Compute per‑channel normalization statistics for ERA5‑based variables used by the model.

**What this notebook does**
1. Reads ERA5 (or your pre‑merged monthly files) in the training period (e.g., 1980–2013).
2. Computes **mean** and **std** per channel; for precipitation, uses **log1p** transform (store as `tp_log`).
3. Saves statistics for later use by sharding, dataloaders, and loss functions.

**Expected outputs**
- `order` — list of channel names in the exact order used.
- `mean` (vector), `std` (vector) — stored in a single `.npz`, e.g., `stats_025deg_1980_2013.npz`.
- (Legacy) Optionally keep `global_means.npy`, `global_stds.npy` for compatibility.

**Tips**
- Enforce lower bounds on std (e.g., `1e-6`). 
- Keep channel order stable across the pipeline.

In [1]:
import os, json, argparse, warnings
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import xarray as xr

In [ ]:
RAW_1DEG  = Path("./data/raw/1deg")
RAW_05DEG = Path("./data/raw/05deg")
RAW_025DEG= Path("./data/raw/025deg")
OUT_DIR   = Path("./data/stats")

SURF_VARS   = ["t2m", "d2m", "msl", "u10", "v10"]
PRECIP_VAR  = "tp"
PRESS_VARS  = ["t","u","v","r","z"]
PRESS_LEVS  = ["850","500","300"]

FEATS = SURF_VARS + ["tp_log"] + [f"{v}_{lev}" for lev in PRESS_LEVS for v in PRESS_VARS]

In [ ]:
# ------------------ utils ------------------
def open_surface_month(root: Path, yr: int, mo: int) -> Optional[xr.Dataset]:
    sf = root / "surface" / f"surface_{yr}_{mo:02d}.nc"
    sfp= root / "surface" / f"surface_p_{yr}_{mo:02d}.nc"
    if not sf.exists() and not sfp.exists():
        return None
    ds = None
    if sf.exists():
        ds = xr.open_dataset(sf)
    if sfp.exists():
        dsp = xr.open_dataset(sfp)
        ds = dsp if ds is None else xr.merge([ds, dsp], compat="override", join="outer")
    return ds

def open_pressure_month(root: Path, yr: int, mo: int) -> Optional[xr.Dataset]:
    pf = root / "pressure_levels" / f"pressure_{yr}_{mo:02d}.nc"
    if not pf.exists():
        return None
    return xr.open_dataset(pf)

def slice_roi(ds: xr.Dataset, roi: Optional[Dict]) -> xr.Dataset:
    if roi is None: 
        return ds
    lat_name = "latitude" if "latitude" in ds.dims else "lat"
    lon_name = "longitude" if "longitude" in ds.dims else "lon"
    lat_min, lat_max = roi["lat_min"], roi["lat_max"]
    lon_min, lon_max = roi["lon_min"], roi["lon_max"]

    if float(ds[lon_name].max()) > 180.0:
        ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360) - 180}).sortby(lon_name)

    lat = ds[lat_name]
    if lat[0] > lat[-1]:
        ds = ds.sel({lat_name: slice(lat_max, lat_min)})
    else:
        ds = ds.sel({lat_name: slice(lat_min, lat_max)})
    ds = ds.sel({lon_name: slice(lon_min, lon_max)})
    return ds

class Welford1D:

    def __init__(self):
        self.count = 0
        self.mean  = 0.0
        self.M2    = 0.0
    def update(self, arr: np.ndarray):
        # arr -> 1D або N-мерний; NaN пропускаємо
        a = arr.ravel()
        m = ~np.isnan(a)
        n = int(m.sum())
        if n == 0: 
            return
        amean = float(a[m].mean())
        ass   = float(((a[m] - amean)**2).sum())
        delta = amean - self.mean
        tot   = self.count + n
        self.mean += delta * n / max(tot, 1)
        self.M2   += ass + (delta**2) * self.count * n / max(tot, 1)
        self.count= tot
    def finalize(self) -> Tuple[float, float, int]:
        var = self.M2 / max(self.count - 1, 1)
        std = float(np.sqrt(max(var, 0.0)))
        return float(self.mean), std, int(self.count)

def ensure_tp_log(ds: xr.Dataset, precip_name: str) -> xr.DataArray:
    if precip_name not in ds:
        raise KeyError(f"'{precip_name}' not in surface dataset variables: {list(ds.data_vars)}")
    tp = ds[precip_name].astype(np.float32)
    tp = xr.where(tp < 0, 0, tp)
    return np.log1p(tp)

def to_levels_int() -> List[int]:
    return [int(l) for l in PRESS_LEVS]

In [ ]:
def compute_stats_for_root(
    root: Path,
    start_year: int,
    end_year: int,
    roi: Optional[Dict] = None,
) -> Dict:
    """
    Return:
      {
        "surface": {var: {"mean": float, "std": float, "count": int}},
        "pressure": {var: {"mean": [L], "std": [L], "count": [L]}},
        "vector": {"order": FEATS, "mean": [K], "std": [K], "count": [K]},
        "meta": {...}
      }
    """
    surf_stats: Dict[str, Welford1D] = {v: Welford1D() for v in SURF_VARS}
    tp_log_stat = Welford1D()
    levs = to_levels_int()
    press_stats: Dict[str, List[Welford1D]] = {v: [Welford1D() for _ in levs] for v in PRESS_VARS}

    for yr in range(start_year, end_year+1):
        for mo in range(1, 12+1):
            sds = open_surface_month(root, yr, mo)
            pds = open_pressure_month(root, yr, mo)
            if sds is None or pds is None:
                continue

            # ROI
            try:
                sds_roi = slice_roi(sds, roi)
                pds_roi = slice_roi(pds, roi)
            except Exception as e:
                warnings.warn(f"ROI slice failed {yr}-{mo:02d} @ {root}: {e}")
                sds_roi, pds_roi = sds, pds

            for v in SURF_VARS:
                if v not in sds_roi:
                    warnings.warn(f"Var '{v}' missing in {root}/surface {yr}-{mo:02d}")
                    continue
                arr = sds_roi[v].values.astype(np.float32)
                surf_stats[v].update(arr)

            try:
                tp_log = ensure_tp_log(sds_roi, PRECIP_VAR)
                tp_log_stat.update(tp_log.values.astype(np.float32))
            except Exception as e:
                warnings.warn(f"tp_log failed {root} {yr}-{mo:02d}: {e}")

            if "pressure_level" not in pds_roi.dims:
                warnings.warn(f"'pressure_level' dim missing in pressure {root} {yr}-{mo:02d}")
                continue
            for li, lev in enumerate(levs):
                sel = pds_roi.sel(pressure_level=lev)
                for v in PRESS_VARS:
                    if v not in sel:
                        warnings.warn(f"Var '{v}' missing at lev={lev} in {root} {yr}-{mo:02d}")
                        continue
                    press_stats[v][li].update(sel[v].values.astype(np.float32))

    surface_out = {}
    for v in SURF_VARS:
        m, s, c = surf_stats[v].finalize()
        surface_out[v] = {"mean": m, "std": s, "count": c}
    m, s, c = tp_log_stat.finalize()
    surface_out["tp_log"] = {"mean": m, "std": s, "count": c}

    pressure_out = {}
    for v in PRESS_VARS:
        means, stds, counts = [], [], []
        for li in range(len(levs)):
            m, s, c = press_stats[v][li].finalize()
            means.append(m); stds.append(s); counts.append(c)
        pressure_out[v] = {"mean": means, "std": stds, "count": counts}

    mean_vec, std_vec, cnt_vec = [], [], []
    for name in FEATS:
        if name == "tp_log":
            mean_vec.append(surface_out["tp_log"]["mean"])
            std_vec.append(surface_out["tp_log"]["std"])
            cnt_vec.append(surface_out["tp_log"]["count"])
        elif name in SURF_VARS:
            mean_vec.append(surface_out[name]["mean"])
            std_vec.append(surface_out[name]["std"])
            cnt_vec.append(surface_out[name]["count"])
        else:
            # pressure name like "t_850"
            v, lev = name.split("_")
            li = PRESS_LEVS.index(lev)
            mean_vec.append(pressure_out[v]["mean"][li])
            std_vec.append(pressure_out[v]["std"][li])
            cnt_vec.append(pressure_out[v]["count"][li])

    res = {
        "surface": surface_out,
        "pressure": pressure_out,
        "vector": {
            "order": FEATS,
            "mean": mean_vec,
            "std": std_vec,
            "count": cnt_vec,
        },
        "meta": {
            "levels": PRESS_LEVS,
            "years": [start_year, end_year],
            "roi": roi,
            "root": str(root.resolve()),
        }
    }
    return res

def save_stats_bundle(res_name: str, stats: Dict, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    npz_path = out_dir / f"stats_{res_name}_{stats['meta']['years'][0]}_{stats['meta']['years'][1]}.npz"
    np.savez_compressed(
        npz_path,
        order=np.array(stats["vector"]["order"], dtype=object),
        mean=np.array(stats["vector"]["mean"], dtype=np.float64),
        std=np.array(stats["vector"]["std"], dtype=np.float64),
        count=np.array(stats["vector"]["count"], dtype=np.int64),
    )
    json_path = out_dir / f"stats_{res_name}_{stats['meta']['years'][0]}_{stats['meta']['years'][1]}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: {npz_path}\n✅ Saved: {json_path}")

In [ ]:
# For CLI and Jupyter compatibility
def parse_args(cli: bool = True, argv=None):
    """
    cli=True  -> reads sys.argv (for console run) and IGNORES unknown args
    cli=False -> DOES NOT read sys.argv (for Jupyter), parses empty list
    """
    ap = argparse.ArgumentParser(description="Compute μ/σ for ERA-like multi-res datasets.")
    ap.add_argument("--y0", type=int, default=1980, help="start year (train)")
    ap.add_argument("--y1", type=int, default=2013, help="end year inclusive (train)")
    ap.add_argument("--roi", type=float, nargs=4, metavar=("LAT_MIN","LAT_MAX","LON_MIN","LON_MAX"),
                    help="optional ROI bbox; if omitted, full domain is used")
    ap.add_argument("--path_1deg", type=str, default=str(RAW_1DEG))
    ap.add_argument("--path_05deg", type=str, default=str(RAW_05DEG))
    ap.add_argument("--path_025deg", type=str, default=str(RAW_025DEG))
    
    if argv is None:
        argv = [] if not cli else None

    args, _ = ap.parse_known_args(argv)
    return args

In [ ]:
args = parse_args(cli=False)
roi = None

jobs = [
        ("1deg",  Path(args.path_1deg)),
        ("05deg", Path(args.path_05deg)),
        ("025deg",Path(args.path_025deg))
        ]

In [8]:
for name, root in jobs:
        if not root.exists():
            warnings.warn(f"Skip {name}: path not found {root}")
            continue
        print(f"\n=== {name}: compute stats {args.y0}-{args.y1} @ {root} ===")
        stats = compute_stats_for_root(root, args.y0, args.y1, roi=roi)
        save_stats_bundle(name, stats, OUT_DIR)


=== 025deg: compute stats 1980-2013 @ data\raw\025deg ===
✅ Saved: data\stats\stats_025deg_1980_2013.npz
✅ Saved: data\stats\stats_025deg_1980_2013.json


In [ ]:
# Check loaded stats
npz = np.load("./data/stats/stats_05deg_1980_2013.npz", allow_pickle=True)
order = list(npz["order"])
mu = npz["mean"]; sigma = npz["std"]

idx = {name:i for i,name in enumerate(order)}

m_t2m, s_t2m = mu[idx["t2m"]], sigma[idx["t2m"]]
m_t_850, s_t_850 = mu[idx["t_850"]], sigma[idx["t_850"]]